# tabular_agent — Phase 3, step 5: reading the model's mistakes

Every earlier step in this series gave the agent a way to *do* something: encode a
column, derive features, train, stop. This one gives it a way to **look at what it
just did wrong**.

Until now the loop had a gap. Map it against the cycle a human data scientist runs —
inspect → hypothesise → build → train → score → decide — and five of the six arcs
existed. The missing one was **score → hypothesise**: `train_tabular_model` writes
every out-of-fold prediction to `experiments/<id>/oof.parquet`, and no tool ever
showed it to anybody. The agent got one number per experiment and had to guess its
next feature from column names. That is why runs kept reaching for `price_squared`:
squaring a column you already have is what "try something" looks like when you
cannot see where you are failing.

`explore_experiment` closes it. This notebook drives it by hand, on a dataset built
so that the failure is real and findable, and only hands the keys to an LLM at the
end.

**What to watch for**, because it is the whole argument of the design record
(`docs/decisions/0020-the-sandbox-has-no-kernel.md`):

1. the aggregate residual looks *fine* while the model is badly, systematically wrong
2. the column that reveals it is one the model **cannot train on**
3. nothing you compute here can become a feature — and that is the only guard there is

> Ships unexecuted. Every code cell was smoke-tested in order first; run it top to
> bottom and the numbers in the prose should be the numbers you see.

## 1. A dataset whose signal is hidden in plain sight

Apartment prices. `area` and `rooms` are numeric and the model will use them.
`district` is a string, so `train_tabular_model` **drops it** — GBDTs take a numeric
matrix, and the sanctioned way to use a categorical column is `fit_encoding`.

The generating rule puts a per-district price *level* on top of area. So the
information the model most needs is sitting in the one column it cannot read. That
is not a contrived setup: it is the state every run starts in, and step 2's notebook
made the same point from the encoder's side.

In [ ]:
import os, shutil, json
import numpy as np, pandas as pd

WORK = os.path.abspath("_step5_demo")
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

rng = np.random.default_rng(11)
n = 900

area = rng.uniform(30, 160, n)
rooms = rng.integers(1, 6, n)
district = rng.choice(["altstadt", "hafen", "vorstadt", "gruenau"], n, p=[.25, .25, .3, .2])

# The whole signal we are going to hunt: a per-district price level the model
# cannot see, because `district` is a string and gets dropped before training.
level = {"altstadt": 2200, "hafen": 1500, "vorstadt": 900, "gruenau": 700}
price = (
    area * np.array([level[d] for d in district])
    + rooms * 4000
    + rng.normal(0, 15000, n)
)

df = pd.DataFrame({"area": area, "rooms": rooms, "district": district, "price": price})
data_path = os.path.join(WORK, "apartments.csv")
df.to_csv(data_path, index=False)
df.head()

`inspect_data` says so out loud — `non_numeric_columns` is the harness telling you
which columns are being left on the table.

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness import tools
from tabular_agent.taskspec import TaskSpec

session = RunSession(os.path.join(WORK, "run"), data_path, TaskSpec(target="price"))
tools.inspect_data(session)["non_numeric_columns"]

## 2. The sandbox learned two things

`explore_experiment` runs your code in the same subprocess sandbox as
`transform_features` — fresh process, import whitelist, wall-clock timeout. Two
changes were needed to point it at run records rather than a single table.

**A namespace instead of one frame.** `run_readonly` now takes `{name: frame}` and
passes your function only the frames it *names*. You declare what you want; you do
not have to declare the rest, or get their order right.

In [ ]:
from tabular_agent.sandbox import run_readonly

frames = {
    "lb": pd.DataFrame({"exp": ["e1", "e2"], "score": [0.31, 0.27]}),
    "notes": pd.DataFrame({"text": ["tried frequency encoding"]}),
}

code = """
def explore(lb):
    return float(lb["score"].min())
"""
run_readonly(frames, code)

A frame name we do not have is a typo or one you invented, so the error lists what
exists rather than dying on a `KeyError` somewhere deeper.

In [ ]:
# A name we do not have is a typo or an invented frame, so the error says what
# exists rather than failing on a KeyError deeper in.
from tabular_agent.sandbox import SandboxError

try:
    run_readonly(frames, "def explore(leaderboard):\n    return 1")
except SandboxError as e:
    print(e)

**Prints come back.** A model writes `print(...)` in exploration code by reflex.
Throwing that output away silently is the same failure as the profile that omitted
category values before step 2 fixed it: the agent behaves sensibly, gets nothing,
and is told nothing about why.

In [ ]:
out = run_readonly(frames, """
def explore(lb, notes):
    print("experiments:", len(lb))
    print(notes["text"].iloc[0])
    return lb["score"].tolist()
""")
print("result:", out["result"])
print("stdout:", repr(out["stdout"]))

And — the half that actually earns its keep — the prints are attached to the
**error** too. Code that fell over on line 8 is fixed using what lines 1–7 printed,
and the exception on its own does not have it.

In [ ]:
# Prints survive the failure path too — which is where they are worth most.
try:
    run_readonly(frames, """
def explore(lb):
    print("about to do something silly")
    return lb["nope"].sum()
""")
except SandboxError as e:
    print(e)

## 3. Train, then look

A first honest model on the raw table. Note `n_features`: two. `district` is gone.

In [ ]:
exp = tools.train_tabular_model(session, model_type="lgbm")
{k: exp[k] for k in ("experiment_id", "val_score", "cv_score", "fold_spread", "n_features")}

The frame `explore_experiment` hands you is the experiment's **whole source table**
— including the string columns the model could not use — plus three columns:

| column | meaning |
|---|---|
| `oof_pred` | what the model predicted for that row, from the fold where it was held out |
| `residual` | `target - oof_pred`, signed |
| `fold` | which CV fold the row sat out in |

Including the unusable columns is a deliberate call. They cost nothing here, because
nothing here can become a feature — and they are the ones worth grouping residuals
by, which is the entire point of what follows.

In [ ]:
tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    return sorted(df.columns)
""")

### The aggregate tells you nothing

Here is the trap. Mean residual ≈ **−47** on prices in the hundreds of thousands.
The errors cancel almost perfectly, and any summary statistic of the whole column
says "unbiased, just noisy".

In [ ]:
tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    return df["residual"].describe().round(1).to_dict()
""")

### Group by the column the model could not use

Same 720 rows, one `groupby`:

```
altstadt   mean  +76,855
hafen      mean  +13,840
vorstadt   mean  −40,086
gruenau    mean  −61,825
```

That is not noise. The model is **systematically** under-pricing `altstadt` by about
77k and over-pricing `gruenau` by about 62k, because it has no way to tell them
apart — so it predicts something near the global level for everything and eats the
difference. The `-47` above was those biases cancelling.

This is the finding no field in any tool result reports, and it took one call.

In [ ]:
by_district = tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    g = df.groupby("district")["residual"].agg(["mean", "std", "size"])
    print("overall |residual| mean:", round(df["residual"].abs().mean(), 1))
    return g.round(1).to_dict("index")
""")
print(by_district["stdout"])
by_district["result"]

### Is it a weak *slice* or one bad *fold*?

Always worth separating, and it is why `fold` is in the frame. Recomputed from
`make_folds` rather than stored — deterministic in `(y, task_type, n_splits, seed)`,
all fixed for the run, so two lines buy the distinction.

Here the error is flat across folds (~47k–52k), so nothing is a fold artifact. The
structure is in the data, not in one unlucky split.

In [ ]:
tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    return df.groupby("fold")["residual"].apply(
        lambda s: float(s.abs().mean())
    ).round(1).to_dict()
""")

## 4. Act on it

The hypothesis is now specific — *encode `district`* — rather than a guess. This is
the arc that was missing: a number came back, it pointed at a column, and the column
suggests a tool call.

`rmsle` **0.485 → 0.221**, a `vs_source_table` gain of `0.264` against a fold spread
of `0.048` — better than five times the run's own noise. Compare that with the two
runs in earlier steps where nineteen experiments found the same score as eight.
Better feedback per cycle beats more cycles.

In [ ]:
enc = tools.fit_encoding(session, ["district"], method="target")
after = tools.train_tabular_model(session, table=enc["table"], model_type="lgbm")
print("before:", round(exp["val_score"], 1))
print("after: ", round(after["val_score"], 1))
print("gain:  ", round(after["vs_source_table"], 1) if after["vs_source_table"] is not None else None)

And the confirmation, from the same tool on the new experiment: the per-district
bias is gone. `+76,855` and `−61,825` have become `+1,226` and `−5`.

Worth noticing what just happened structurally — the diagnostic that *found* the
problem is also the one that shows it is fixed, and neither call cost an experiment.

In [ ]:
tools.explore_experiment(session, after["experiment_id"], """
def explore(df):
    g = df.groupby("district")["residual"].agg(["mean", "size"])
    return g.round(1).to_dict("index")
""")

## 5. What it cannot do

Three limits, and they are what make a free-form code tool acceptable inside a
harness whose whole claim is "leakage-safe by construction".

**It cannot train.** Same import whitelist as `transform_features`. Nothing that
learns state is importable.

In [ ]:
# No model library — exploration cannot fit anything, here or anywhere.
try:
    tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    import lightgbm
    return 1
""")
except SandboxError as e:
    print(e)

**There is no kernel.** A name bound in one call is gone in the next.

This was the real design fork, and it went the other way from the obvious answer.
The tempting model — borrowed from prime-agent's "RLM" — is a *persistent* Python
kernel where the run's records live in variables and only what you print enters the
context. The context economics are excellent. But they do not come from the kernel
being alive; they come from the sandbox boundary, which we already had. What
persistence adds on top is names that survive between calls, and that is the one
kind of state this project has spent every decision removing: 0003 put the tables on
disk, 0004 the record, 0010 the transform's own source code, and `RunSession`
promises that a fresh process handed the same `run_dir` continues the run.

A kernel variable is run state that is **not** in `run_dir` — it cannot be resumed
and cannot be read after a crash. And the planned episode loop runs a fresh agent
per episode against the same run directory, so those variables would not survive the
seam anyway.

In [ ]:
# No kernel. A name bound in one call is gone in the next: the state is the run
# directory, and a resumed run must behave identically to this one.
tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    global stash
    stash = df["residual"].tolist()
    return "stashed"
""")

try:
    tools.explore_experiment(session, exp["experiment_id"], "def explore(df):\n    return len(stash)")
except SandboxError as e:
    print(e)

**Nothing full-length leaves.** The result is capped at 200 rows. Below is the exact
shape of a hand-rolled target encoding — a value per row, computed on all the data.
It comes back truncated, so it is not a usable feature vector.

Be clear about what this does and does not prove. The `price` column *is* in the
frame, and you can look at per-district target means all day. The guard is not that
the number is hidden — it is that **there is no path to persist it**. The only route
from a category→target signal to an actual column stays `fit_encoding`, which fits
per fold. And per 0010 a transform is stored as *code* and re-run at predict, so a
hardcoded literal pasted back would die on a length mismatch the first time it met
new rows.

The honest residual risk is written down in 0020: an agent choosing features by
staring at out-of-fold errors is doing selection informed by held-out predictions.
That leak runs through the model's reasoning, not through the data, and no guard in
this system catches it.

In [ ]:
# A full-length column cannot leave. This is the shape of a hand-rolled target
# encoding, and the cap is what stops it from being a usable feature vector.
out = tools.explore_experiment(session, exp["experiment_id"], """
def explore(df):
    return df.groupby("district")["price"].transform("mean")
""")
r = out["result"]
print(r["__kind__"], "| n_rows:", r["n_rows"], "| truncated:", r["truncated"], "| returned:", len(r["data"]))

**It is free, and it records nothing.** No leaderboard row, no budget spent — same
deal as `explain_table`, for the same reason: charging for a look at a failure would
price diagnosis the same as causing one.

In [ ]:
# Free, and it records nothing: same leaderboard before and after.
from tabular_agent.harness import journal

before = len(journal.load(session.run_dir)["leaderboard"])
for _ in range(3):
    tools.explore_experiment(session, exp["experiment_id"], "def explore(df):\n    return 1")
print(before, "->", len(journal.load(session.run_dir)["leaderboard"]))

It also still works after `finalize_run`. The moment you most want to ask what went
wrong is the moment you have just been told to stop.

In [ ]:
report = tools.finalize_run(session)
# Still available after the run is finished — the moment you most want to ask
# what went wrong is when you have just been told to stop.
tools.explore_experiment(session, exp["experiment_id"], "def explore(df):\n    return len(df)")

## 6. The surface, now seven tools

That is the whole thing the model can reach. Two of them run code the agent wrote;
the difference between them is entirely what happens to the output — one becomes
columns and is replayed at predict, the other is shown once and thrown away.

In [ ]:
from tabular_agent.harness.agent import build_agent
agent = build_agent("openai:gpt-4o")
sorted(agent._function_toolset.tools)

## 7. Handing it to a model

Everything above was driven by hand on purpose. The tool is registered, so a live
agent can now do the same thing — and the interesting question for the next run is
whether it *will*: whether it reaches for `explore_experiment` after a disappointing
score, or keeps reaching for `price_squared`.

The docstring is the only place it is told what is in the frame (PydanticAI ships
each tool's docstring as its description), so that docstring is worth reading as
prompt text rather than as documentation.

Needs `config.env` with a `REQUESTY_API_KEY`. In a notebook use `await agent.run(...)`
— `run_sync` raises inside a running event loop.

In [ ]:
from tabular_agent.harness.agent import build_agent, gateway_model, DEFAULT_GATEWAY_MODEL, Deps

PROMPT = """Build the best model you can for this dataset. When a score
disappoints you, look at where the model is actually wrong before deciding what to
try next."""

live = RunSession(os.path.join(WORK, "run_llm"), data_path, TaskSpec(target="price"))
agent = build_agent(gateway_model(DEFAULT_GATEWAY_MODEL))

result = await agent.run(PROMPT, deps=Deps(session=live))
print(result.output)

### Where this leaves the sequence

`explore_journal` is the sibling this notebook does not build: the same sandbox
pointed at the run's *records* (`lb`, `tables`, `notes`) instead of one experiment's
rows. The namespace plumbing in §2 exists for it. It was deferred because its
justification is a 150-experiment run — at eight experiments the leaderboard is
cheaper to read as text than to query — and 150 experiments needs **episodes**,
which do not exist yet. That is the next thing.